# CS3807 Deep Learning Laboratory - Part 2: Attention Implementations
## Implementing Channel, Spatial, and Self-Attention Mechanisms

This notebook implements all attention mechanisms required for the lab.

In [ ]:
# Cell 1: Import libraries (same as Part 1)
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tqdm import tqdm

# Load previous data
with open('results/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']
class_names = splits['class_names']
num_classes = splits['num_classes']
train_augment = splits['train_augment']

print(f"Data loaded successfully")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## SECTION 1: CHANNEL ATTENTION IMPLEMENTATION

In [ ]:
# Cell 2: Implement Channel Attention mechanism
class ChannelAttention(layers.Layer):
    """
    Channel Attention Module
    Formula: M_c(F) = σ(W_2 δ(W_1 z))
    where z = (1/HW) Σ F(i,j,c) [global average pooling]
    δ = ReLU, σ = Sigmoid
    """
    def __init__(self, reduction_ratio=16, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        
    def build(self, input_shape):
        channels = input_shape[-1]
        reduced_channels = max(channels // self.reduction_ratio, 1)
        
        self.global_avg_pool = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1 = layers.Dense(reduced_channels, activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        
        super(ChannelAttention, self).build(input_shape)
        
    def call(self, inputs):
        # Global average pooling: F(H,W,C) -> (1,1,C)
        channel_desc = self.global_avg_pool(inputs)  # z_c
        
        # Learnable transformation through FC layers
        channel_weights = self.fc1(channel_desc)      # W_1 z_c (reduction)
        channel_weights = self.fc2(channel_weights)   # W_2 δ(W_1 z_c) (restoration + sigmoid)
        
        # Element-wise multiplication: F ⊙ M_c(F)
        output = inputs * channel_weights
        
        return output

print("✓ Channel Attention class defined")
print("  Formula: M_c(F) = σ(W_2 δ(W_1 z_c))")
print("  Reduction ratio: 16 (default)")
print("  Output shape: Same as input")

In [ ]:
# Cell 3: Test Channel Attention implementation
# Create a simple test input
test_input = np.random.randn(1, 14, 14, 1024).astype(np.float32)  # P3 feature map

channel_attn = ChannelAttention(reduction_ratio=16)
test_output = channel_attn(test_input)

print("Channel Attention Test:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Shape preserved: {test_input.shape == test_output.shape}")
print(f"  Trainable parameters: {channel_attn.count_params()}")
print("  ✓ Channel Attention working correctly")

## SECTION 2: SPATIAL ATTENTION IMPLEMENTATION

In [ ]:
# Cell 4: Implement Spatial Attention mechanism
class SpatialAttention(layers.Layer):
    """
    Spatial Attention Module
    Formula: M_s(F) = σ(Conv_7×7([F_avg ∥ F_max]))
    where F_avg = Mean_c(F), F_max = Max_c(F)
    σ = Sigmoid
    """
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        padding = kernel_size // 2
        
        self.conv = layers.Conv2D(
            filters=1,
            kernel_size=kernel_size,
            padding='same',
            activation='sigmoid'
        )
        
    def call(self, inputs):
        # Channel-wise pooling
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)  # F_avg
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)   # F_max
        
        # Concatenate: [F_avg ∥ F_max]
        concat = tf.concat([avg_pool, max_pool], axis=-1)  # Shape: (B,H,W,2)
        
        # 7x7 Convolution + Sigmoid
        spatial_weights = self.conv(concat)  # Shape: (B,H,W,1)
        
        # Element-wise multiplication: F ⊙ M_s(F)
        output = inputs * spatial_weights
        
        return output

print("✓ Spatial Attention class defined")
print("  Formula: M_s(F) = σ(Conv_7×7([F_avg ∥ F_max]))")
print("  Kernel size: 7×7 (default)")
print("  Output shape: Same as input")

In [ ]:
# Cell 5: Test Spatial Attention implementation
spatial_attn = SpatialAttention(kernel_size=7)
test_output_spatial = spatial_attn(test_input)

print("Spatial Attention Test:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output_spatial.shape}")
print(f"  Shape preserved: {test_input.shape == test_output_spatial.shape}")
print(f"  Trainable parameters: {spatial_attn.count_params()}")
print("  ✓ Spatial Attention working correctly")

## SECTION 3: SEQUENTIAL CHANNEL-SPATIAL ATTENTION

In [ ]:
# Cell 6: Implement Sequential Channel-Spatial Attention
class ChannelSpatialAttention(layers.Layer):
    """
    Sequential Channel-Spatial Attention
    Flow: F -> Channel Attention -> Spatial Attention -> F_cs
    Formula: F_c = M_c(F) ⊙ F
             F_cs = M_s(F_c) ⊙ F_c
    """
    def __init__(self, reduction_ratio=16, spatial_kernel=7, **kwargs):
        super(ChannelSpatialAttention, self).__init__(**kwargs)
        self.channel_attn = ChannelAttention(reduction_ratio=reduction_ratio)
        self.spatial_attn = SpatialAttention(kernel_size=spatial_kernel)
        
    def call(self, inputs):
        # Step 1: Apply channel attention
        x = self.channel_attn(inputs)  # F_c = M_c(F) ⊙ F
        
        # Step 2: Apply spatial attention to channel-attended features
        x = self.spatial_attn(x)       # F_cs = M_s(F_c) ⊙ F_c
        
        return x

print("✓ Sequential Channel-Spatial Attention class defined")
print("  Flow: F -> Channel Attn -> Spatial Attn -> F_cs")
print("  This module applies channel attention FIRST, then spatial attention")

In [ ]:
# Cell 7: Test Sequential Channel-Spatial Attention
cs_attn = ChannelSpatialAttention(reduction_ratio=16, spatial_kernel=7)
test_output_cs = cs_attn(test_input)

print("Channel-Spatial Attention Test:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output_cs.shape}")
print(f"  Shape preserved: {test_input.shape == test_output_cs.shape}")
print(f"  Trainable parameters: {cs_attn.count_params()}")
print("  ✓ Channel-Spatial Attention working correctly")

## SECTION 4: SELF-ATTENTION IMPLEMENTATION

In [ ]:
# Cell 8: Implement Self-Attention mechanism
class SelfAttention(layers.Layer):
    """
    Lightweight Self-Attention Module
    Formula: A = softmax(QK^T / √d)
             Y = AV
             F_SA = F + γ reshape(Y)
    """
    def __init__(self, dim_scale=8, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.dim_scale = dim_scale  # Reduces computational cost
        
    def build(self, input_shape):
        channels = input_shape[-1]
        self.scaled_dim = max(channels // self.dim_scale, 1)
        
        # Projections for Q, K, V
        self.query_proj = layers.Dense(self.scaled_dim)
        self.key_proj = layers.Dense(self.scaled_dim)
        self.value_proj = layers.Dense(self.scaled_dim)
        self.output_proj = layers.Dense(channels)
        
        # Learnable scaling parameter (residual)
        self.gamma = tf.Variable(initial_value=tf.zeros((1,)), trainable=True)
        
        super(SelfAttention, self).build(input_shape)
        
    def call(self, inputs):
        # inputs shape: (B, H, W, C)
        batch_size = tf.shape(inputs)[0]
        H, W, C = inputs.shape[1], inputs.shape[2], inputs.shape[3]
        
        # Reshape to (B, N, C) where N = HW
        x_reshaped = tf.reshape(inputs, [batch_size, H*W, C])
        
        # Generate Query, Key, Value
        Q = self.query_proj(x_reshaped)  # (B, N, scaled_dim)
        K = self.key_proj(x_reshaped)    # (B, N, scaled_dim)
        V = self.value_proj(x_reshaped)  # (B, N, scaled_dim)
        
        # Compute attention: A = softmax(QK^T / √d)
        attention_scores = tf.matmul(Q, K, transpose_b=True)  # (B, N, N)
        attention_scores = attention_scores / tf.math.sqrt(tf.cast(self.scaled_dim, tf.float32))
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        
        # Apply attention to values: Y = A V
        attended_values = tf.matmul(attention_weights, V)  # (B, N, scaled_dim)
        attended_values = self.output_proj(attended_values)  # (B, N, C)
        
        # Reshape back to spatial: (B, H, W, C)
        attended_values = tf.reshape(attended_values, [batch_size, H, W, C])
        
        # Residual connection with learnable scaling: F_SA = F + γ Y
        output = inputs + self.gamma * attended_values
        
        return output

print("✓ Self-Attention class defined")
print("  Formula: A = softmax(QK^T / √d), Y = AV")
print("  Residual: F_SA = F + γ reshape(Y)")
print("  Dimension scale factor: 8 (reduces computation)")
print("  NOTE: Use only at P3 or P4 (high spatial resolution = expensive)")

In [ ]:
# Cell 9: Test Self-Attention implementation
self_attn = SelfAttention(dim_scale=8)
test_output_self = self_attn(test_input)

print("Self-Attention Test:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output_self.shape}")
print(f"  Shape preserved: {test_input.shape == test_output_self.shape}")
print(f"  Trainable parameters: {self_attn.count_params()}")
print("  ✓ Self-Attention working correctly")
print("\n  Note: Attention matrix size = N×N where N=HW")
print(f"    For P3 (14×14): N = 196, A size = 196×196")
print(f"    For P2 (28×28): N = 784, A size = 784×784 (EXPENSIVE!)")

## SECTION 5: COMBINED ATTENTION MODULES

In [ ]:
# Cell 10: Implement Reverse Attention (Spatial->Channel for Experiment 4)
class SpatialChannelAttention(layers.Layer):
    """
    Reverse Sequential Attention
    Flow: F -> Spatial Attention -> Channel Attention -> F_sc
    Compare with Channel->Spatial
    """
    def __init__(self, reduction_ratio=16, spatial_kernel=7, **kwargs):
        super(SpatialChannelAttention, self).__init__(**kwargs)
        self.spatial_attn = SpatialAttention(kernel_size=spatial_kernel)
        self.channel_attn = ChannelAttention(reduction_ratio=reduction_ratio)
        
    def call(self, inputs):
        # Step 1: Apply spatial attention FIRST
        x = self.spatial_attn(inputs)  # F_s = M_s(F) ⊙ F
        
        # Step 2: Apply channel attention to spatially-attended features
        x = self.channel_attn(x)       # F_sc = M_c(F_s) ⊙ F_s
        
        return x

print("✓ Reverse Spatial-Channel Attention class defined")
print("  Flow: F -> Spatial Attn -> Channel Attn -> F_sc")
print("  This is the OPPOSITE order of Channel-Spatial")
print("  Used for Experiment 4: Attention Order Ablation")

In [ ]:
# Cell 11: Create flexible Spatial Attention with variable kernel sizes
class VariableSpatialAttention(layers.Layer):
    """
    Spatial Attention with configurable kernel size
    Used for Experiment 5: Kernel Size Ablation
    """
    def __init__(self, kernel_size=7, **kwargs):
        super(VariableSpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        
        self.conv = layers.Conv2D(
            filters=1,
            kernel_size=kernel_size,
            padding='same',
            activation='sigmoid'
        )
        
    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        spatial_weights = self.conv(concat)
        output = inputs * spatial_weights
        return output

print("✓ Variable Spatial Attention class defined")
print("  Used for Experiment 5: Kernel Size Ablation")
print("  Kernel sizes to test: 3×3, 5×5, 7×7")

In [ ]:
# Cell 12: Implement Combined Attention (Channel + Spatial + Self)
class CombinedAttention(layers.Layer):
    """
    Combined Attention Module
    Applies Channel, Spatial, and Self-Attention sequentially
    """
    def __init__(self, reduction_ratio=16, spatial_kernel=7, dim_scale=8, **kwargs):
        super(CombinedAttention, self).__init__(**kwargs)
        self.channel_attn = ChannelAttention(reduction_ratio=reduction_ratio)
        self.spatial_attn = SpatialAttention(kernel_size=spatial_kernel)
        self.self_attn = SelfAttention(dim_scale=dim_scale)
        
    def call(self, inputs):
        x = self.channel_attn(inputs)
        x = self.spatial_attn(x)
        x = self.self_attn(x)
        return x

print("✓ Combined Attention (Channel + Spatial + Self) class defined")
print("  Flow: F -> Channel -> Spatial -> Self-Attention -> F_combined")
print("  Used for Experiment 1: Model A5")

## SECTION 6: SQUEEZE-AND-EXCITATION (SE) ATTENTION

In [ ]:
# Cell 13: Implement SE Attention (Reference: Hu et al., 2020)
class SEAttention(layers.Layer):
    """
    Squeeze-and-Excitation Attention
    Similar to channel attention but with specific design choices
    Reference: Hu et al., 2020 - "Squeeze-and-Excitation Networks"
    """
    def __init__(self, reduction_ratio=16, **kwargs):
        super(SEAttention, self).__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        
    def build(self, input_shape):
        channels = input_shape[-1]
        reduced_channels = max(channels // self.reduction_ratio, 1)
        
        self.global_avg_pool = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1 = layers.Dense(reduced_channels, activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        
        super(SEAttention, self).build(input_shape)
        
    def call(self, inputs):
        # Squeeze: Global average pooling
        squeeze = self.global_avg_pool(inputs)
        
        # Excitation: FC layers with ReLU and Sigmoid
        excitation = self.fc1(squeeze)
        excitation = self.fc2(excitation)
        
        # Scale: Element-wise multiplication
        output = inputs * excitation
        
        return output

print("✓ SE Attention class defined (Extended Experiment)")
print("  Similar to Channel Attention but specifically designed as SE module")
print("  Reference: Hu et al., 2020")

In [ ]:
# Cell 14: Implement ECA Attention (Reference: Wang et al., 2020)
class ECAAttention(layers.Layer):
    """
    Efficient Channel Attention (ECA)
    GAP -> 1D conv across channels (adaptive kernel) -> sigmoid -> scale.
    Adaptive kernel: k = |log2(C)/gamma + b/gamma|_odd, gamma=2, b=1.
    Reference: Wang et al., 2020 - 'ECA-Net: Efficient Channel Attention'
    """
    def __init__(self, gamma=2, b=1, **kwargs):
        super(ECAAttention, self).__init__(**kwargs)
        self.gamma = gamma
        self.b = b
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
    def _adaptive_kernel(self, channels):
        import math
        k = int(abs((math.log2(channels) / self.gamma) + (self.b / self.gamma)))
        return k if k % 2 == 1 else k + 1
    def build(self, input_shape):
        channels = int(input_shape[-1])
        k = self._adaptive_kernel(channels)
        self.conv1d = layers.Conv1D(1, kernel_size=k, padding='same', use_bias=False,
                                  activation='sigmoid')
        super(ECAAttention, self).build(input_shape)
    def call(self, inputs):
        z = self.gap(inputs)                      # (B,1,1,C)
        z = tf.squeeze(z, axis=1)                 # (B,1,C)
        w = self.conv1d(z)                        # (B,1,C) cross-channel interaction
        w = tf.expand_dims(w, axis=1)             # (B,1,1,C)
        return inputs * w

print("✓ ECA Attention class defined")
print("  Flow: GAP -> adaptive 1D-conv -> sigmoid -> scale")
print("  Used for Extended Experiment: additional-mechanism comparison")

In [ ]:
# Cell 15: Implement CBAM (Reference: Woo et al., 2018)
class CBAM(layers.Layer):
    """
    Convolutional Block Attention Module (CBAM)
    Channel submodule (avg+max pool -> shared MLP -> sigmoid), then
    spatial submodule (avg+max along channels -> 7x7 conv -> sigmoid).
    Reference: Woo et al., 2018 - 'CBAM: Convolutional Block Attention Module'
    """
    def __init__(self, reduction_ratio=16, spatial_kernel=7, **kwargs):
        super(CBAM, self).__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.spatial_kernel = spatial_kernel
    def build(self, input_shape):
        channels = int(input_shape[-1])
        reduced = max(channels // self.reduction_ratio, 1)
        self.mlp_fc1 = layers.Dense(reduced, activation='relu')
        self.mlp_fc2 = layers.Dense(channels)
        self.spatial_conv = layers.Conv2D(1, kernel_size=self.spatial_kernel,
                                        padding='same', activation='sigmoid')
        super(CBAM, self).build(input_shape)
    def call(self, inputs):
        avg_c = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
        max_c = tf.reduce_max(inputs, axis=[1, 2], keepdims=True)
        mlp_avg = self.mlp_fc2(self.mlp_fc1(avg_c))
        mlp_max = self.mlp_fc2(self.mlp_fc1(max_c))
        ch_w = tf.nn.sigmoid(mlp_avg + mlp_max)
        x = inputs * ch_w
        avg_s = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_s = tf.reduce_max(x, axis=-1, keepdims=True)
        sp_w = self.spatial_conv(tf.concat([avg_s, max_s], axis=-1))
        return x * sp_w

print("✓ CBAM class defined (channel -> spatial)")
print("  Used for Extended Experiment: additional-mechanism comparison")

In [ ]:
# Cell 16: Implement Coordinate Attention (Reference: Hou et al., 2021)
class CoordinateAttention(layers.Layer):
    """
    Coordinate Attention: factorises spatial attention into two 1D
    descriptors (H-pool and W-pool), transforms them jointly, then
    re-weights features along each axis separately.
    Reference: Hou et al., 2021 - 'Coordinate Attention for Efficient Mobile Network Design'
    """
    def __init__(self, reduction=32, **kwargs):
        super(CoordinateAttention, self).__init__(**kwargs)
        self.reduction = reduction
    def build(self, input_shape):
        channels = int(input_shape[-1])
        mid = max(channels // self.reduction, 8)
        self.conv_reduce = layers.Conv2D(mid, 1, padding='same', use_bias=False)
        self.bn = layers.BatchNormalization()
        self.conv_h = layers.Conv2D(channels, 1, padding='same', activation='sigmoid')
        self.conv_w = layers.Conv2D(channels, 1, padding='same', activation='sigmoid')
        super(CoordinateAttention, self).build(input_shape)
    def call(self, inputs):
        zh = tf.reduce_mean(inputs, axis=2, keepdims=True)   # pool over W -> (B,H,1,C)
        zw = tf.reduce_mean(inputs, axis=1, keepdims=True)   # pool over H -> (B,1,W,C)
        zw_t = tf.transpose(zw, [0, 2, 1, 3])                # (B,W,1,C)
        z = tf.concat([zh, zw_t], axis=1)
        z = self.conv_reduce(z)
        z = self.bn(z)
        z = tf.nn.relu(z)
        zh_f, zw_f = tf.split(z, [tf.shape(zh)[1], tf.shape(zw_t)[1]], axis=1)
        zw_f = tf.transpose(zw_f, [0, 2, 1, 3])
        gh = self.conv_h(zh_f)
        gw = self.conv_w(zw_f)
        return inputs * gh * gw

print("✓ Coordinate Attention class defined")
print("  Flow: H-pool + W-pool -> shared 1x1 -> split -> sigmoid gates")
print("  Used for Extended Experiment: additional-mechanism comparison")

In [ ]:
# Cell 17: Save all attention module classes for reuse
attention_modules = {
    'ChannelAttention': ChannelAttention,
    'SpatialAttention': SpatialAttention,
    'ChannelSpatialAttention': ChannelSpatialAttention,
    'SpatialChannelAttention': SpatialChannelAttention,
    'VariableSpatialAttention': VariableSpatialAttention,
    'SelfAttention': SelfAttention,
    'CombinedAttention': CombinedAttention,
    'SEAttention': SEAttention,
    'ECAAttention': ECAAttention,
    'CBAM': CBAM,
    'CoordinateAttention': CoordinateAttention
}

print("Summary of Implemented Attention Modules:")
print("="*60)
for name in attention_modules.keys():
    print(f"  ✓ {name}")

print("\nAll attention mechanisms ready for model construction.")

---

**Attention Implementation Complete!**

**Summary:**
- ✓ Channel Attention: Learns importance of feature channels
- ✓ Spatial Attention: Learns importance of spatial locations
- ✓ Sequential Channel-Spatial: Channel first, then spatial
- ✓ Reverse Spatial-Channel: Spatial first, then channel (Exp. 4)
- ✓ Variable Spatial: Kernel size 3×3, 5×5, 7×7 (Exp. 5)
- ✓ Self-Attention: Spatial interaction learning
- ✓ Combined: All three sequentially
- ✓ SE Attention: Squeeze-and-Excitation (Extended)
- ✓ ECA Attention: Efficient channel attention, adaptive 1D conv (Extended)
- ✓ CBAM: Channel + spatial submodules (Extended)
- ✓ Coordinate Attention: Direction-aware H/W gates (Extended)

**Next: Part 3 - Model Construction and Training**